# Soul Manteca — Generador de imagen de producto

Recrea a una persona caminando por la calle con el **mismo conjunto Reebok** de tu foto, usando dos motores de IA (Google Gemini y OpenAI).

**Cómo usarlo:** ejecútá las celdas **una por una, de arriba hacia abajo**, tocando el botón ▶️ (play) a la izquierda de cada una. Solé esperar a que termine cada una antes de la siguiente.

## Paso 1 — Instalar lo necesario
Tocá ▶️. Tarda ~20 segundos. Cuando aparece un ✅ está listo.

In [ ]:
!pip install -q google-genai requests
print('✅ Listo, seguí con el Paso 2')

## Paso 2 — Subí la foto del conjunto Reebok
Al tocar ▶️ aparece un botón **"Elegir archivos"**. Seleccioná la foto del conjunto (la del piso). Se guarda como referencia.

In [ ]:
import os
from google.colab import files
os.makedirs('referencia', exist_ok=True)
os.makedirs('salidas', exist_ok=True)
print('Elegí la foto del conjunto Reebok:')
subido = files.upload()
nombre = list(subido.keys())[0]
RUTA_REF = 'referencia/reebok.jpeg'
with open(RUTA_REF, 'wb') as f:
    f.write(subido[nombre])
print(f'✅ Foto guardada ({len(subido[nombre])} bytes). Seguí con el Paso 3')

## Paso 3 — Pegá tus claves (API keys)
Al tocar ▶️ van a aparecer dos casillas. Pegá cada clave y apretá Enter. **No se ven** mientras escribís (es por seguridad) y **no quedan guardadas**.

- Clave de **Google (Gemini)**: la sacás en https://aistudio.google.com/apikey
- Clave de **OpenAI**: la sacás en https://platform.openai.com/api-keys

👉 Si solo tenés una de las dos, dejá la otra **vacía** (Enter sin pegar nada) y ese motor se saltea.

In [ ]:
from getpass import getpass
os.environ['GEMINI_API_KEY'] = getpass('Pegá tu clave de Google (Gemini) y Enter: ').strip()
os.environ['OPENAI_API_KEY'] = getpass('Pegá tu clave de OpenAI y Enter: ').strip()
print('✅ Claves cargadas. Seguí con el Paso 4')

## Paso 4 — Generar las imágenes
Tocá ▶️ y esperá. Puede tardar entre 20 segundos y 1–2 minutos. Las imágenes aparecen acá abajo y además se descargan solas a tu teléfono/compu.

In [ ]:
import base64, binascii, mimetypes, requests
from datetime import datetime
from pathlib import Path
from IPython.display import Image as Show, display

PROMPT = (
    "Full-body photo of a young person walking through a gritty urban street in "
    "Buenos Aires, wearing this exact navy blue Reebok nylon tracksuit (zip jacket "
    "with grey side stripes, matching jogger pants with elastic cuffs). Keep the "
    "garment identical to the reference. Overcast natural light, slight 90s-2000s "
    "grain, shallow depth of field, confident stride, concrete walls and stencil "
    "graffiti in the background. Photorealistic, editorial street style."
)

REF = Path('referencia/reebok.jpeg')
MIME = mimetypes.guess_type(str(REF))[0] or 'image/jpeg'
img_bytes = REF.read_bytes()

def nombre(motor, ext='png'):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return Path('salidas') / f'soul_manteca_{motor}_{ts}.{ext}'

def decodificar(v):
    if isinstance(v, (bytes, bytearray)):
        if v[:8].startswith(b'\x89PNG') or v[:3] == b'\xff\xd8\xff':
            return bytes(v)
        try:
            return base64.b64decode(v, validate=True)
        except (binascii.Error, ValueError):
            return bytes(v)
    return base64.b64decode(v)

salidas = []

# ---- Motor 1: Gemini ----
if os.environ.get('GEMINI_API_KEY'):
    print('\n[Gemini] Generando...')
    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        r = client.models.generate_content(
            model='gemini-3.1-flash-image-preview',
            contents=[types.Part.from_bytes(data=img_bytes, mime_type=MIME), PROMPT],
        )
        guardada = None
        for cand in (getattr(r, 'candidates', None) or []):
            for part in (getattr(getattr(cand, 'content', None), 'parts', None) or []):
                inline = getattr(part, 'inline_data', None)
                if inline and getattr(inline, 'data', None):
                    dest = nombre('gemini')
                    dest.write_bytes(decodificar(inline.data))
                    guardada = dest
                    break
            if guardada:
                break
        if guardada:
            print(f'[Gemini] ✅ {guardada}')
            salidas.append(guardada)
        else:
            print('[Gemini] ⚠️ No devolvió imagen.', getattr(r, 'text', '') or '')
    except Exception as e:
        print(f'[Gemini] ❌ Error: {type(e).__name__}: {e}')
else:
    print('[Gemini] Salteado (sin clave).')

# ---- Motor 2: OpenAI ----
if os.environ.get('OPENAI_API_KEY'):
    print('\n[OpenAI] Generando...')
    try:
        with REF.open('rb') as fh:
            resp = requests.post(
                'https://api.openai.com/v1/images/edits',
                headers={'Authorization': f"Bearer {os.environ['OPENAI_API_KEY']}"},
                files={'image': (REF.name, fh, MIME)},
                data={'model': 'gpt-image-2', 'prompt': PROMPT, 'size': '1024x1536', 'n': '1'},
                timeout=300,
            )
        if resp.status_code == 200:
            dest = nombre('openai')
            dest.write_bytes(decodificar(resp.json()['data'][0]['b64_json']))
            print(f'[OpenAI] ✅ {dest}')
            salidas.append(dest)
        else:
            try:
                msg = resp.json().get('error', {}).get('message', resp.text)
            except Exception:
                msg = resp.text
            if resp.status_code == 403 and any(s in msg.lower() for s in ('verif', 'organization')):
                print('[OpenAI] ⚠️ Tu organización no está verificada para gpt-image-2.')
                print('         Verificala en https://platform.openai.com/settings/organization/general')
            print(f'[OpenAI] ❌ HTTP {resp.status_code}: {msg}')
    except Exception as e:
        print(f'[OpenAI] ❌ Error: {type(e).__name__}: {e}')
else:
    print('[OpenAI] Salteado (sin clave).')

# ---- Mostrar y descargar ----
print('\n== Resultado ==')
if salidas:
    for s in salidas:
        print(s)
        display(Show(filename=str(s)))
        files.download(str(s))
else:
    print('No se generó ninguna imagen. Revisá los mensajes de arriba.')